# 🧹 Preprocesamiento — Detección de toxicidad en YouTube (`IsToxic`)

## 🎯 Objetivo
Pipeline completo de limpieza, lematización, split y data augmentation sobre
`youtoxic_english_1000.csv` con variable objetivo `IsToxic`.

## ⚠️ Decisión clave: split ANTES del augmentation
El augmentation se aplica **únicamente sobre el conjunto de entrenamiento** tras el split.
Si se hace antes, versiones sintéticas de textos del train pueden acabar en test →
métricas infladas → evaluación inválida en producción.

## 📋 Flujo del notebook
1. Setup e imports
2. Carga y selección de columnas
3. Limpieza con regex
4. Lematización con spaCy
5. Split estratificado train / val / test
6. Data Augmentation solo sobre train
7. Exportación de CSVs a `data/processed/V_04`

In [3]:
import re
import os
import time
import warnings

import pandas as pd
import numpy as np
import nltk
import spacy
from tqdm import tqdm
from deep_translator import GoogleTranslator
import nlpaug.augmenter.word as naw
from sklearn.model_selection import train_test_split

warnings.filterwarnings('ignore')

# Recursos NLTK necesarios
for resource in ['punkt', 'punkt_tab', 'stopwords', 'wordnet',
                 'omw-1.4', 'averaged_perceptron_tagger_eng']:
    nltk.download(resource, quiet=True)

# Crear carpeta de salida si no existe
os.makedirs('../data/processed/V_04', exist_ok=True)

TARGET       = 'IsToxic'
RANDOM_STATE = 42
AUG_THRESHOLD = 0.48  # ratio mínimo de clase tóxica en train antes de augmentar

print('Setup completado.')
print(f'Carpeta de salida: ../data/processed/V_04')

Setup completado.
Carpeta de salida: ../data/processed/V_04


## 📂 1. Carga del dataset y selección de columnas

Se cargan solo las columnas necesarias:
- `CommentId` y `VideoId` para trazabilidad
- `Text` como entrada de texto
- `IsToxic` como variable objetivo

In [5]:
df_raw = pd.read_csv('../../data/raw/youtoxic_english_1000.csv')

print('Shape completo:', df_raw.shape)
print('Columnas      :', df_raw.columns.tolist())
print('\nNulos:\n', df_raw.isnull().sum())

df = df_raw[['CommentId', 'VideoId', 'Text', TARGET]].copy()

print('\nShape tras selección de columnas:', df.shape)
print('\nDistribución de IsToxic:')
print(df[TARGET].value_counts())
print(df[TARGET].value_counts(normalize=True).round(3))

Shape completo: (1000, 15)
Columnas      : ['CommentId', 'VideoId', 'Text', 'IsToxic', 'IsAbusive', 'IsThreat', 'IsProvocative', 'IsObscene', 'IsHatespeech', 'IsRacist', 'IsNationalist', 'IsSexist', 'IsHomophobic', 'IsReligiousHate', 'IsRadicalism']

Nulos:
 CommentId          0
VideoId            0
Text               0
IsToxic            0
IsAbusive          0
IsThreat           0
IsProvocative      0
IsObscene          0
IsHatespeech       0
IsRacist           0
IsNationalist      0
IsSexist           0
IsHomophobic       0
IsReligiousHate    0
IsRadicalism       0
dtype: int64

Shape tras selección de columnas: (1000, 4)

Distribución de IsToxic:
IsToxic
False    538
True     462
Name: count, dtype: int64
IsToxic
False    0.538
True     0.462
Name: proportion, dtype: float64


## 🧹 2. Limpieza de texto con expresiones regulares

Pipeline aplicado en orden:
1. Minúsculas
2. Eliminación de URLs (`http`, `www`, `https`)
3. Eliminación de menciones (`@usuario`) y hashtags (`#tag`)
4. Eliminación de dígitos
5. Solo letras y espacios
6. Normalización de espacios múltiples

Al final se detectan y eliminan filas cuyo texto quedó vacío tras la limpieza.

In [6]:
def clean_text(text):
    if not isinstance(text, str):
        return ''
    text = text.lower()
    text = re.sub(r'http\S+|www\S+|https\S+', '', text, flags=re.MULTILINE)
    text = re.sub(r'@\w+|#\w+', '', text)
    text = re.sub(r'\d+', '', text)
    text = re.sub(r'[^a-z\s]', '', text)
    text = re.sub(r'\s+', ' ', text).strip()
    return text

# Ejemplo paso a paso
ejemplo = df['Text'].iloc[1]
print('Original :', ejemplo)
print('Limpio   :', clean_text(ejemplo))

df['clean_text'] = df['Text'].apply(clean_text)

# Detectar y eliminar textos vacíos
vacios = (df['clean_text'] == '').sum()
print(f'\nTextos vacíos tras limpieza: {vacios}')
if vacios > 0:
    df = df[df['clean_text'] != ''].reset_index(drop=True)
    print(f'Shape tras eliminar vacíos: {df.shape}')

# Estadísticas de reducción
antes   = df['Text'].str.len().mean()
despues = df['clean_text'].str.len().mean()
print(f'\nLongitud media antes  : {antes:.0f} caracteres')
print(f'Longitud media después: {despues:.0f} caracteres')
print(f'Reducción             : {(1 - despues/antes)*100:.1f}%')

df[['Text', 'clean_text']].head(3)

Original : Law enforcement is not trained to shoot to apprehend.  They are trained to shoot to kill.  And I thank Wilson for killing that punk bitch.
Limpio   : law enforcement is not trained to shoot to apprehend they are trained to shoot to kill and i thank wilson for killing that punk bitch

Textos vacíos tras limpieza: 2
Shape tras eliminar vacíos: (998, 5)

Longitud media antes  : 186 caracteres
Longitud media después: 177 caracteres
Reducción             : 4.8%


,Text,clean_text
0,If only people would just take a step back and...,if only people would just take a step back and...
1,Law enforcement is not trained to shoot to app...,law enforcement is not trained to shoot to app...
2,\r\nDont you reckon them 'black lives matter' ...,dont you reckon them black lives matter banner...
